In [ ]:
import pandas as pd
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, LpBinary
from itertools import product
from collections import defaultdict

# Load dataset
file_path = "datasets_v5/realdata_v5_2.xlsx"
activities_df = pd.read_excel(file_path, sheet_name="ActivitiesInfo")
classroom_df = pd.read_excel(file_path, sheet_name="ClassroomsInfo")
student_courses_df = pd.read_excel(file_path, sheet_name="StudentsInfo")

In [132]:
student_courses_df.head()

,Student_ID,Class_ID,Course_Names,Course_ID,Activity_Names,Activity_IDs
0,S0001,26,"Research Skills for Financial Mathematics, Num...","53, 50, 28, 5",MATHF Research Skills for Financial Mathematic...,"5301, 5001, 2801, 2802, 2803, 2804, 501, 502, 503"
1,S0002,19,"Algebraic Geometry, Probability, Measure & Fin...","41, 7, 50, 4","MATH5 Algebraic Geometry - on-site exam, MATH4...","4101, 701, 702, 703, 704, 5001, 401, 402"
2,S0003,26,"Research Skills for Financial Mathematics, Num...","53, 50, 28",MATHF Research Skills for Financial Mathematic...,"5301, 5001, 2801, 2802, 2803, 2804"
3,S0004,5,"Topics in Mathematical Physics A, Essentials i...","51, 8, 6, 41","MATH5 Topics in Maths Physics A - Workshop1, M...","5101, 5102, 801, 601, 602, 603, 4101"
4,S0005,10,"Introduction to Lie Groups, Research Skills fo...","19, 47, 20, 31","MATH5 Introduction to Lie Groups - Workshop1, ...","1901, 1902, 1903, 4701, 4702, 4703, 2001, 2002..."


### Model 1：

In [46]:
# Index set
courses = student_courses_df["Course_ID"]
courses = set([int(num) for row in courses for num in row.split(', ')])

classes = student_courses_df["Class_ID"]
classes = set([row for row in classes])

classrooms = classroom_df["Classroom_ID"]
classrooms = set(int(row) for row in classrooms)

weeks = classroom_df["Available_Weeks"]
weeks = set([int(num) for row in weeks for num in row.split(', ')])

time_slots = classroom_df["Time_Slot"]
time_slots = set([int(num) for row in time_slots for num in row.split(', ')])

activities = activities_df["Activity_ID"]
activities = set([int(row) for row in activities])

activity_types = activities_df["Activity_Type"]
activity_types = set(activity_types)

In [47]:
for _, row in activities_df.iterrows():
    row_prefer_weeks = [int(i) for i in str(row["Preferred_Weeks"]).split(', ')]
    row_available_weeks = [int(i) for i in str(row["Available_Weeks"]).split(', ')]
    if set(row_available_weeks) != set(row_prefer_weeks):
        print('1', row["Course_ID"], row["Activity_ID"], row["Activity_Type"], row["Num_Students"])


In [ ]:
# Parameters
B = ["morning", "afternoon"]
Gsa = [(row["Course_ID"], row["Activity_ID"]) for _, row in activities_df.iterrows() if row["Requires_Separation"] == 1]
Ysai = [(row["Course_ID"], row["Activity_ID"], row["Activity_Type"]) for _, row in activities_df.iterrows()]
Csa = [(row["Course_ID"], row["Activity_ID"]) for _, row in activities_df.iterrows() if row["Requires_Computers"] == 1]
Tsa = [(row["Course_ID"], row["Activity_ID"]) for _, row in activities_df.iterrows() if row["Requires_Tables"] == 1]
Bsab = [(row["Course_ID"], row["Activity_ID"], row["Preferred_Time"]) for _, row in activities_df.iterrows() if row["Preferred_Time"] in B]

Gamma_sa = [(row["Course_ID"], row["Activity_ID"]) for _, row in activities_df.iterrows() if row["Activity_Type"] in [2, 3]]
Ltb_morning = [(i, "morning") for i in range(1, 4)]
Ltb_afternoon = [(i, "afternoon") for i in range(4, 10)]
Ltb = Ltb_morning + Ltb_afternoon
Ltb_mapping = {row[0]: row[1] for row in Ltb}
Ltb_mapping_inv = {row[1]: row[0] for row in Ltb}
Omega_tt = {}
for t1 in time_slots:
    for t2 in time_slots:
        if Ltb_mapping.get(t1, 0) == Ltb_mapping.get(t2, 0):
            Omega_tt[(t1, t2)] = 1
max_courses_per_week = 7
Psak = {(row["Course_ID"], row["Activity_ID"], k): int(week) 
        for _, row in activities_df.iterrows() 
        for k, week in enumerate(str(row["Preferred_Weeks"]).split(', ')[:3])}
Wk = {k: penalty for k, penalty in zip(range(3), [0, 0.5, 0.7])}

activity_type_mapping = {row["Activity_ID"]: row["Activity_Type"] for _, row in activities_df.iterrows()}
Msa = [(row["Course_ID"], row["Activity_ID"]) for _, row in activities_df.iterrows() if row["Duration"] == '02:00']

In [ ]:
# Parameters for reducing dimensions
available_weeks = {(row["Course_ID"], row["Activity_ID"]): str(row["Available_Weeks"]).split(', ') for _, row in activities_df.iterrows()}
available_weeks = {k: set([int(vv) for vv in v]) for k, v in available_weeks.items()}
sa = [(s, a) for s in courses for a in activities_df[activities_df["Course_ID"] == s]["Activity_ID"]]
sa_dict = defaultdict(list)
for s, a in sa:
    sa_dict[s].append(a)

week_courses = {(int(week), row["Course_ID"]) for _, row in activities_df.iterrows() for week in [i for i in str(row["Available_Weeks"]).split(', ')]}
week_courses_dict = defaultdict(list)
for week, s in week_courses:
    week_courses_dict[week].append(s)

week_courses_dict = dict(week_courses_dict)

for s, a in sa_dict.items():
    if len(set(a)) != len(a):
        print(s, a)

valid_time_pairs = {(t, t_prime) for t in time_slots for t_prime in time_slots if t != t_prime}

valid_conflict_pairs = [(s, a, b, b_prime) for s in courses for a in sa_dict[s] 
                        for b in B for b_prime in B if b != b_prime 
                        and (s, a, b) in Bsab and (s, a, b_prime) in Bsab]

valid_pairs = [(s, a) for s in courses for a in sa_dict[s] if (s, a) in Gsa and (s, a) in Tsa]

x_re = [(w, t, s, a) for s in courses for a in sa_dict[s] for w in available_weeks[(s, a)] for t in time_slots]

for t in time_slots:
    for i in range(len(valid_pairs)):
        for j in range(i + 1, len(valid_pairs)):
            for w in available_weeks[(valid_pairs[i][0], valid_pairs[i][1])] & available_weeks[(valid_pairs[j][0], valid_pairs[j][1])]:
                if valid_pairs[i] != valid_pairs[j]:
                    if (w, t, valid_pairs[i][0], valid_pairs[i][1]) not in x_re:
                        print('1')
                    if (w, t, valid_pairs[j][0], valid_pairs[j][1]) not in x_re:
                        print('2')

In [ ]:
# Define model
model = LpProblem(name="course_scheduling", sense=LpMinimize)

lambda_R = 0.1
Wk = {k: penalty for k, penalty in zip(range(3), [0, 1000, 1300])}


# Define variables
y = {(w, s, a): LpVariable(f"y_{w}_{s}_{a}", cat=LpBinary) for s in courses for a in sa_dict[s] for w in available_weeks[(s, a)]}
x = {(w, t, s, a): LpVariable(f"x_{w}_{t}_{s}_{a}", cat=LpBinary) 
     for s in courses for a in sa_dict[s] for w in available_weeks[(s, a)] for t in time_slots}

# Objective function
model += (
    lpSum(Wk[k] * y[Psak[s, a, k], s, a] for k in range(1, 3) for s in courses for a in sa_dict[s] if Psak.get((s, a, k), -1) != -1)
    + lambda_R * lpSum(
        (x[w, t, valid_pairs[i][0], valid_pairs[i][1]] + x[w, t, valid_pairs[j][0], valid_pairs[j][1]] - 1)
        for t in time_slots
        for i in range(len(valid_pairs))
        for j in range(i + 1, len(valid_pairs))
        for w in available_weeks[(valid_pairs[i][0], valid_pairs[i][1])] & available_weeks[(valid_pairs[j][0], valid_pairs[j][1])]
        if valid_pairs[i] != valid_pairs[j]
    )
), "Minimize_Preference_Penalty_And_Soft_Conflicts"


# Constraints A.2 - A.11
# A.2
for s in courses:
    for a in sa_dict[s]:
        model += lpSum(x[w, t, s, a] for t in time_slots for w in available_weeks[(s, a)]) >= 1, f"Ensure_One_Assignment_{s}_{a}"

# A.3
for s in courses:
    for a in sa_dict[s]:
        for b in B:
            for w in available_weeks[(s, a)]:
                model += lpSum(
                    x[w, t, s, a] for t in time_slots if (t, b) in Ltb
                ) <= 1, f"Max_One_Per_Limited_Slot_{s}_{a}_{b}_{w}"

# A.4
for s in courses:
    for a in sa_dict[s]:
        for b in B:
            if (s, a, b) not in Bsab:
                for w in available_weeks[(s, a)]:
                    model += lpSum(x[w, t, s, a] for t in time_slots if (t, b) in Ltb) == 0, f"Forbidden_Resource_{s}_{a}_{b}_{w}"

# A.5
for s in courses:
    for a in sa_dict[s]:
        for t, t_prime in valid_time_pairs:
            if Omega_tt.get((t, t_prime), 0) == 0 or (s, a) not in Gsa:
                for w in available_weeks[(s, a)]:
                    model += x[w, t, s, a] + x[w, t_prime, s, a] <= y[w, s, a], \
                                f"Time_Slot_Limit_{w}_{s}_{a}_{t}_{t_prime}"

# A.6（modified）
for s, a, b, b_prime in valid_conflict_pairs:
    for w in available_weeks[(s, a)]: 
        model += (
            lpSum(x[w, t, s, a] for t in Ltb_mapping_inv[b]) +
            lpSum(x[w, t_prime, s, a] for t_prime in Ltb_mapping_inv[b_prime])
        ) >= 2 * y[w, s, a], f"Resource_Conflict_{w}_{s}_{a}_{b}_{b_prime}"


# A.7
for w in weeks:
    model += lpSum(
        y[w, s, a] for s in week_courses_dict[w] for a in sa_dict[s] if (s, a) in Gamma_sa and w in available_weeks[(s, a)]
    ) <= max_courses_per_week, f"Max_Courses_Per_Student_{w}"

# new constraint
for s in courses:
    for a in sa_dict[s]:
        for w in available_weeks[(s, a)]:
            for w_prime in available_weeks[(s, a)]:
                if w != w_prime:
                    model += (
                        y[w, s, a] + y[w_prime, s, a]
                        ) <= 1, f"Every_Activity_Once_A_Week_{s}_{a}_{w}_{w_prime}"

# Solver
model.solve()

w_schedule_results = [
    {"Week": w, "Course_ID": s, "Activity_ID": a, "Probability": y[w, s, a].value()}
    for w, s, a in y if y[w, s, a].value() >= 0.5
]

w_t_schedule_results = [
    {"Week": w, "Time_slot": t, "Course_ID": s, "Activity_ID": a, "Probability": x[w, t, s, a].value()}
    for w, t, s, a in x if x[w, t, s, a].value() >= 0.5
]

w_schedule_df = pd.DataFrame(w_schedule_results)

w_t_schedule_df = pd.DataFrame(w_t_schedule_results)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/envs/pp2/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/th/3vtt05957_l2_90vqr26g_dr0000gn/T/c6fe1c64abc54863a501174339e10779-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/th/3vtt05957_l2_90vqr26g_dr0000gn/T/c6fe1c64abc54863a501174339e10779-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 27149 COLUMNS
At line 131601 RHS
At line 158746 BOUNDS
At line 164867 ENDATA
Problem MODEL has 27144 rows, 6120 columns and 87888 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 589.5 - 0.05 seconds
Cgl0002I 2700 variables fixed
Cgl0003I 0 fixed, 0 tightened bounds, 3803 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 2 tightened bounds, 3705 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 2 tightened bounds, 3135 strengthened

In [ ]:
count = 0
count_prior = 0
for _, row in w_schedule_df.iterrows():
    course_id, activity_id = int(row["Course_ID"]), int(row["Activity_ID"])
    count += 1
    if Psak[(course_id, activity_id, 0)] == int(row["Week"]):
        count_prior += 1
    # print(course_id, activity_id, Psak[(course_id, activity_id, 0)], int(row["Week"]), Psak[(course_id, activity_id, 0)] == int(row["Week"]))
print(count, count_prior, count - count_prior)

204 147 57


In [98]:
w_schedule_df

,Week,Course_ID,Activity_ID,Probability
0,2,1,101,1.0
1,2,1,102,1.0
2,2,1,103,1.0
3,4,1,104,1.0
4,4,1,105,1.0
...,...,...,...,...
199,11,52,5205,1.0
200,1,53,5301,1.0
201,4,53,5302,1.0
202,7,53,5303,1.0


In [84]:
w_t_schedule_df

,Week,Time_slot,Course_ID,Activity_ID,Probability
0,2,3,1,101,1.0
1,2,1,1,102,1.0
2,2,1,1,103,1.0
3,4,7,1,104,1.0
4,4,8,1,105,1.0
...,...,...,...,...,...
199,11,1,52,5205,1.0
200,1,7,53,5301,1.0
201,4,8,53,5302,1.0
202,7,5,53,5303,1.0


##### Result processing
1. ** Sort the activity candidate set by (week, time slot) ** : For example, week 1 time slot 1, candidate set: {"course1", "course3",... }. (The question assumes that only one activity can be held per subject per week)

2. ** Activities that remove online exams **

3. ** Result Backup ** : csv format is as follows
|week|time_slot|subject_candidates|

4. ** Run Model 2 ** once per time point per week

In [ ]:
def process_data(df):
    df = df[~df["Activity_ID"].map(activity_type_mapping).eq(3)]
    grouped = df.groupby(["Week", "Time_slot"])["Course_ID"].apply(lambda x: set(x)).reset_index()
    grouped.rename(columns={"Course_ID": "subject_candidates"}, inplace=True)
    grouped.to_csv("processed_schedule_0.1.csv", index=False)
    
    return grouped
subject_candidates = process_data(w_t_schedule_df)
subject_candidates

,Week,Time_slot,subject_candidates
0,1,1,"{35, 40, 11, 15, 18}"
1,1,2,"{35, 11, 43, 13, 48, 52}"
2,1,3,"{35, 5, 40, 11, 16}"
3,1,4,"{48, 35}"
4,1,5,"{2, 47}"
...,...,...,...
74,12,2,{2}
75,12,3,"{48, 33}"
76,12,5,{18}
77,12,7,{30}
